# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [ ]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

## 2. Pipeline de procesamiento

In [ ]:
feature_columns = contract["features"]

# Preprocesador para árboles: solo imputación por mediana (no requieren escalado)
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), feature_columns),
    ]
)

# Preprocesador para modelos sensibles a escala: imputación + StandardScaler
scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), feature_columns),
    ]
)

print("Preprocesadores creados.")

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Si ajustamos el `StandardScaler` sobre todo el conjunto de entrenamiento antes de la validación cruzada, la media y desviación estándar se calculan usando también los datos que luego servirán como fold de validación. Esto es **data leakage**: información del fold de validación se filtra al preprocesamiento. Al poner el scaler dentro del pipeline, en cada fold de la CV el scaler se ajusta únicamente con el fold de entrenamiento y se aplica (transform) al fold de validación, dando una estimación honesta del desempeño.

## 3. Modelos candidatos

In [ ]:
models = {
    "decision_tree": Pipeline([
        ("prep", tree_preprocessor),
        ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("prep", scaled_preprocessor),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("prep", scaled_preprocessor),
        ("clf", KNeighborsClassifier()),
    ]),
}

print("Modelos candidatos:", list(models.keys()))

## 4. Comparación inicial con validación cruzada

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

initial_comparison = []
for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
    )
    initial_comparison.append({
        "model": name,
        "train_accuracy_mean": scores["train_score"].mean(),
        "val_accuracy_mean": scores["test_score"].mean(),
        "val_accuracy_std": scores["test_score"].std(),
    })

initial_comparison = pd.DataFrame(initial_comparison).sort_values(
    "val_accuracy_mean", ascending=False
)
initial_comparison

## 5. Optimización de hiperparámetros

In [ ]:
# Espacios de búsqueda pequeños para cada modelo
param_grids = {
    "decision_tree": {
        "clf__max_depth": [3, 5, 7, None],
        "clf__min_samples_leaf": [1, 2, 4],
    },
    "logistic_regression": {
        "clf__C": [0.1, 1.0, 10.0],
    },
    "knn": {
        "clf__n_neighbors": [3, 5, 7, 9],
        "clf__weights": ["uniform", "distance"],
    },
}

searches = {}
tuned_comparison = []

for name, model in models.items():
    search = GridSearchCV(
        model,
        param_grids[name],
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    tuned_comparison.append({
        "model": name,
        "best_cv_accuracy": search.best_score_,
        "best_params": search.best_params_,
    })

tuned_comparison = pd.DataFrame(tuned_comparison).sort_values(
    "best_cv_accuracy", ascending=False
)
tuned_comparison

## 6. Selección y persistencia del campeón

In [ ]:
# Seleccionar el mayor best_score_ de CV
champion_name = max(searches, key=lambda n: searches[n].best_score_)
champion_search = searches[champion_name]
champion = champion_search.best_estimator_

print("Campeón:", champion_name)
print("CV accuracy:", champion_search.best_score_)
print("Mejores hiperparámetros:", champion_search.best_params_)

# Guardar champion_model.joblib
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

# Guardar training_metadata.json
training_metadata = {
    "champion": champion_name,
    "cv_accuracy": float(champion_search.best_score_),
    "best_params": {k: (v if not isinstance(v, (np.integer, np.floating)) else float(v))
                    for k, v in champion_search.best_params_.items()},
    "random_state": RANDOM_STATE,
    "candidates": tuned_comparison["model"].tolist(),
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, default=str),
    encoding="utf-8",
)

# Guardar cv_results.csv (resumen de la comparación con tuning)
cv_results = tuned_comparison.copy()
cv_results["best_params"] = cv_results["best_params"].astype(str)
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)

print("\nArtefactos guardados.")

In [ ]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

**Justificación de la selección:**

> El modelo campeón se elige por tener la mayor accuracy en validación cruzada. En el dataset Wine, tanto la regresión logística como KNN (con escalado) suelen alcanzar accuracy muy alta (>0.97) con baja variabilidad entre folds, mientras que el árbol de decisión tiende a sobreajustar (accuracy de train cercano a 1.0 pero menor en validación). Se prefiere el modelo con alto desempeño en validación, baja desviación estándar (estabilidad) y menor brecha train-validation (menos sobreajuste). La regresión logística además ofrece buena interpretabilidad mediante sus coeficientes.

## Checklist

- [x] Todo el procesamiento está dentro de pipelines.
- [x] Comparé tres familias de modelos.
- [x] Optimicé hiperparámetros con CV.
- [x] Elegí el campeón sin usar test.
- [x] Guardé el pipeline completo y sus metadatos.